In [ ]:
import pandas as pd
import numpy as np

cohort = pd.read_pickle(
    "../results/cohort_features_checkpoint.pkl"
)

# All categorical predictors from the verified paper map
all_categorical_cols = [
    "RIAGENDR",          # Gender
    "RIDRETH1",          # Race
    "INDHHINC",          # Income
    "CVDFITLV",          # Cardiovascular fitness level
    "PAQ180",            # Physical activity
    "BPQ020",            # Hypertension history
    "XY_Estimation",     # DNAm estimated sex
    "MCQ160B",           # Heart failure history
    "MCQ160C",           # Coronary artery disease history
    "MCQ160F",           # Cerebrovascular/stroke history
    "DERIVED_HL_MED",    # Hyperlipidemia medication
    "DERIVED_HTN_MED",   # Hypertension medication
    "DERIVED_DM_MED",    # Diabetes medication
    "DIQ010",            # Diabetes history
]

# Only use columns that currently exist
categorical_cols = [
    col for col in all_categorical_cols
    if col in cohort.columns
]

missing_categorical = [
    col for col in all_categorical_cols
    if col not in cohort.columns
]

print("Categorical columns available now:")
print(categorical_cols)

print("\nStill need to build later:")
print(missing_categorical)

Categorical columns available now:
['RIAGENDR', 'RIDRETH1', 'INDHHINC', 'CVDFITLV', 'PAQ180', 'BPQ020', 'MCQ160B', 'MCQ160C', 'MCQ160F', 'DIQ010']

Still need to build later:
['XY_Estimation', 'DERIVED_HL_MED', 'DERIVED_HTN_MED', 'DERIVED_DM_MED']


In [ ]:
cohort_encoded = pd.get_dummies(
    cohort,
    columns=categorical_cols,
    dummy_na=False,
    dtype=int
)

print("Before one-hot:", cohort.shape)
print("After one-hot:", cohort_encoded.shape)

Before one-hot: (9957, 265)
After one-hot: (9957, 303)


In [ ]:
for col in categorical_cols:
    new_cols = [
        c for c in cohort_encoded.columns
        if c.startswith(col + "_")
    ]

    print(col, "->", new_cols)

RIAGENDR -> ['RIAGENDR_1.0', 'RIAGENDR_2.0']
RIDRETH1 -> ['RIDRETH1_1.0', 'RIDRETH1_2.0', 'RIDRETH1_3.0', 'RIDRETH1_4.0', 'RIDRETH1_5.0']
INDHHINC -> ['INDHHINC_1.0', 'INDHHINC_2.0', 'INDHHINC_3.0', 'INDHHINC_4.0', 'INDHHINC_5.0', 'INDHHINC_6.0', 'INDHHINC_7.0', 'INDHHINC_8.0', 'INDHHINC_9.0', 'INDHHINC_10.0', 'INDHHINC_11.0', 'INDHHINC_12.0', 'INDHHINC_13.0', 'INDHHINC_77.0', 'INDHHINC_99.0']
CVDFITLV -> ['CVDFITLV_1.0', 'CVDFITLV_2.0', 'CVDFITLV_3.0']
PAQ180 -> ['PAQ180_1.0', 'PAQ180_2.0', 'PAQ180_3.0', 'PAQ180_4.0']
BPQ020 -> ['BPQ020_1.0', 'BPQ020_2.0', 'BPQ020_9.0']
MCQ160B -> ['MCQ160B_1.0', 'MCQ160B_2.0', 'MCQ160B_7.0', 'MCQ160B_9.0']
MCQ160C -> ['MCQ160C_1.0', 'MCQ160C_2.0', 'MCQ160C_7.0', 'MCQ160C_9.0']
MCQ160F -> ['MCQ160F_1.0', 'MCQ160F_2.0', 'MCQ160F_9.0']
DIQ010 -> ['DIQ010_1.0', 'DIQ010_2.0', 'DIQ010_3.0', 'DIQ010_7.0', 'DIQ010_9.0']


In [ ]:
cohort_ohe = cohort.copy()

special_missing = {
    "INDHHINC": [77, 99],
    "PAQ180": [7, 9],
    "BPQ020": [7, 9],
    "MCQ160B": [7, 9],
    "MCQ160C": [7, 9],
    "MCQ160F": [7, 9],
    "DIQ010": [7, 9],
}

for col, codes in special_missing.items():
    if col in cohort_ohe.columns:
        cohort_ohe[col] = cohort_ohe[col].replace(codes, np.nan)

In [ ]:
cohort_encoded = pd.get_dummies(
    cohort_ohe,
    columns=categorical_cols,
    dummy_na=False,
    dtype=int
)

print("Before one-hot:", cohort_ohe.shape)
print("After one-hot:", cohort_encoded.shape)

Before one-hot: (9957, 265)
After one-hot: (9957, 293)


In [ ]:
bad_dummy_cols = [
    c for c in cohort_encoded.columns
    if any(
        c.endswith(suffix)
        for suffix in [
            "_7.0",
            "_9.0",
            "_77.0",
            "_99.0"
        ]
    )
]

print("Invalid dummy columns:", bad_dummy_cols)

Invalid dummy columns: ['INDHHINC_7.0', 'INDHHINC_9.0']


In [ ]:
[c for c in cohort.columns if "mort" in c.lower() or "death" in c.lower()]

['MORTSTAT', 'death_5yr']

In [ ]:
cohort_encoded["mortality_5yr"] = (
    (cohort_encoded["MORTSTAT"] == 1) &
    (cohort_encoded["PERMTH_INT"] <= 60)
).astype(int)

print(cohort_encoded["mortality_5yr"].value_counts())

mortality_5yr
0    8664
1    1293
Name: count, dtype: int64


In [ ]:
y = cohort_encoded["mortality_5yr"]

majority_class = y.mode()[0]

y_pred_stupid = np.full(
    len(y),
    majority_class
)

accuracy = (y_pred_stupid == y).mean()

print("Majority class:", majority_class)
print(f"Stupid model accuracy: {accuracy:.2%}")

Majority class: 0
Stupid model accuracy: 87.01%


In [ ]:
print(
    "Targets identical:",
    (cohort_encoded["death_5yr"] ==
     cohort_encoded["mortality_5yr"]).all()
)

Targets identical: True


In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

y = cohort_encoded["death_5yr"].astype(int)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(np.zeros(len(y)), y),
    start=1
):
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Learn majority class ONLY from this training fold
    majority_class = y_train.mode().iloc[0]

    # Predict same class for everyone in validation fold
    y_pred = np.full(
        len(y_val),
        majority_class
    )

    accuracy = (
        y_pred == y_val.to_numpy()
    ).mean()

    fold_accuracies.append(accuracy)

    print(
        f"Fold {fold}: "
        f"{accuracy:.2%}"
    )

print(
    "\nMean 5-fold accuracy:",
    f"{np.mean(fold_accuracies):.2%}"
)

print(
    "Std:",
    f"{np.std(fold_accuracies):.4f}"
)

Fold 1: 87.00%
Fold 2: 87.00%
Fold 3: 87.04%
Fold 4: 87.04%
Fold 5: 86.99%

Mean 5-fold accuracy: 87.01%
Std: 0.0002


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import numpy as np

In [ ]:
y = cohort["death_5yr"].astype(int)

# Never use outcome/linkage fields as predictors
drop_cols = [
    "SEQN",
    "death_5yr",
    "MORTSTAT",
    "PERMTH_INT"
]

X = cohort.drop(
    columns=[c for c in drop_cols if c in cohort.columns]
)

categorical_cols = [
    c for c in all_categorical_cols
    if c in X.columns
]

numeric_cols = [
    c for c in X.columns
    if c not in categorical_cols
]

print("Categorical:", len(categorical_cols))
print("Numeric:", len(numeric_cols))

Categorical: 10
Numeric: 251


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    # Learn preprocessing ONLY from training fold
    X_train_processed = preprocessor.fit_transform(X_train)

    # Apply same learned preprocessing to validation
    X_val_processed = preprocessor.transform(X_val)

    print(
        f"Fold {fold}:",
        "train =", X_train_processed.shape,
        "| validation =", X_val_processed.shape
    )

/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL' 'DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL' 'DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 1: train = (7965, 295) | validation = (1992, 295)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 2: train = (7965, 297) | validation = (1992, 297)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 3: train = (7966, 297) | validation = (1991, 297)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 4: train = (7966, 297) | validation = (1991, 297)
Fold 5: train = (7966, 296) | validation = (1991, 296)


/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMDEDUC3' 'DMDSCHOL']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:
all_missing_cols = [
    col for col in X.columns
    if X[col].isna().all()
]

print("Completely empty columns:", len(all_missing_cols))
print(all_missing_cols)

Completely empty columns: 2
['DMDEDUC3', 'DMDSCHOL']


In [ ]:
X = X.drop(columns=all_missing_cols)

categorical_cols = [
    c for c in all_categorical_cols
    if c in X.columns
]

numeric_cols = [
    c for c in X.columns
    if c not in categorical_cols
]

print("Predictors remaining:", X.shape[1])
print("Categorical:", len(categorical_cols))
print("Numeric:", len(numeric_cols))

Predictors remaining: 259
Categorical: 10
Numeric: 249


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)

    print(
        f"Fold {fold}:",
        "train =", X_train_processed.shape,
        "| validation =", X_val_processed.shape
    )

/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/evodemanirahari/Documents/nhanes-cardiovascular-ml/.venv/lib/python3.14/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['DMAETHN' 'DMARACE']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


Fold 1: train = (7965, 295) | validation = (1992, 295)
Fold 2: train = (7965, 297) | validation = (1992, 297)
Fold 3: train = (7966, 297) | validation = (1991, 297)
Fold 4: train = (7966, 297) | validation = (1991, 297)
Fold 5: train = (7966, 296) | validation = (1991, 296)


In [1]:
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Load data
cohort = pd.read_pickle("../results/cohort_features_checkpoint.pkl")

# 2. Select three features
glm_features = ["RIDAGEYR", "SSTNT", "SSBNP"]

X_glm = cohort[glm_features].copy()
y_glm = cohort["death_5yr"].astype(int)

# 3. Build Logistic Regression
glm = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# 4. Stratified 5-fold cross-validation
glm_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 5. Evaluate
glm_scores = cross_validate(
    glm,
    X_glm,
    y_glm,
    cv=glm_cv,
    scoring={
        "Precision": "precision",
        "Recall": "recall",
        "F1": "f1",
        "ROC-AUC": "roc_auc"
    },
    error_score="raise"
)

# 6. Display results
glm_results = pd.DataFrame({
    "Precision": glm_scores["test_Precision"],
    "Recall": glm_scores["test_Recall"],
    "F1": glm_scores["test_F1"],
    "ROC-AUC": glm_scores["test_ROC-AUC"]
})

glm_results.index = range(1, 6)

print("Logistic Regression - 5-Fold Results")
display(glm_results.round(3))

print("\nAverage Performance:")
display(glm_results.mean().round(3))

Logistic Regression - 5-Fold Results


,Precision,Recall,F1,ROC-AUC
1,0.282,0.741,0.409,0.792
2,0.292,0.749,0.420,0.802
3,0.291,0.764,0.421,0.821
4,0.325,0.829,0.467,0.852
5,0.288,0.772,0.420,0.814



Average Performance:


Precision    0.296
Recall       0.771
F1           0.427
ROC-AUC      0.816
dtype: float64

In [ ]:
## Random Forest Model

Compare Random Forest with Logistic Regression using
the same three features and stratified 5-fold cross-validation.